# Computing Text Similarity on the Validation Split

This notebook demonstrates the course evaluation method using the provided validation labels and example validation predictions. It matches rows by `image_id`, embeds text with `sentence-transformers/all-MiniLM-L6-v2`, L2-normalizes the 384-dimensional embeddings, and reports mean cosine similarity.

Replace `Validation_Predictions_Example.csv` with your own validation prediction file to evaluate your model. This embedding model is for evaluation only and must not be used to fit the caption generator.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")

DATA_ROOT = Path.cwd()
if not (DATA_ROOT / "Validation_Labels.csv").is_file():
    candidate = DATA_ROOT / "AIAA2205Project"
    if candidate.is_dir():
        DATA_ROOT = candidate
if not (DATA_ROOT / "Validation_Labels.csv").is_file():
    raise FileNotFoundError("Run this notebook from AIAA2205Project/ or its parent.")

LABELS_PATH = DATA_ROOT / "Validation_Labels.csv"
PREDICTIONS_PATH = DATA_ROOT / "Validation_Predictions_Example.csv"
print(f"Data root: {DATA_ROOT}")

Python and package versions printed by the current environment.


In [2]:
labels = pd.read_csv(LABELS_PATH)
predictions = pd.read_csv(PREDICTIONS_PATH)

if list(labels.columns) != ["image_filename", "description"]:
    raise ValueError(f"Unexpected label columns: {list(labels.columns)}")
if list(predictions.columns) != ["image_id", "description"]:
    raise ValueError(f"Unexpected prediction columns: {list(predictions.columns)}")

labels = labels.assign(image_id=labels["image_filename"].map(lambda value: Path(value).stem))
if labels["image_id"].duplicated().any() or predictions["image_id"].duplicated().any():
    raise ValueError("Duplicate image IDs found.")
if predictions["description"].fillna("").str.strip().eq("").any():
    raise ValueError("Predictions contain blank descriptions.")

merged = labels[["image_id", "description"]].rename(
    columns={"description": "reference_description"}
).merge(
    predictions.rename(columns={"description": "predicted_description"}),
    on="image_id",
    how="inner",
    validate="one_to_one",
)
if len(merged) != len(labels) or len(merged) != len(predictions):
    raise ValueError("Prediction IDs do not exactly match validation IDs.")
print(f"Matched rows: {len(merged)}")

Matched rows: 2500


In [3]:
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

model_dir = DATA_ROOT / "Text_Embedding_Model"
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
embedding_model = AutoModel.from_pretrained(model_dir, local_files_only=True)
embedding_model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedding_model.to(device)

@torch.inference_mode()
def encode_texts(texts, batch_size=64):
    batches = []
    for start in range(0, len(texts), batch_size):
        tokens = tokenizer(
            texts[start:start + batch_size], padding=True, truncation=True,
            max_length=64, return_tensors="pt",
        )
        tokens = {name: tensor.to(device) for name, tensor in tokens.items()}
        hidden = embedding_model(**tokens).last_hidden_state
        mask = tokens["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1e-9)
        batches.append(F.normalize(pooled.float(), p=2, dim=1).cpu())
    return torch.cat(batches).numpy()

reference_embeddings = encode_texts(merged["reference_description"].tolist())
prediction_embeddings = encode_texts(merged["predicted_description"].tolist())
if reference_embeddings.shape[1] != 384:
    raise ValueError(f"Expected 384-dimensional embeddings, got {reference_embeddings.shape}")

per_image_cosine = np.sum(reference_embeddings * prediction_embeddings, axis=1)
mean_cosine = float(per_image_cosine.mean())
print(f"Mean validation cosine similarity: {mean_cosine:.6f}")

Mean validation cosine similarity: 0.583020
